In [ ]:
import random
import torch
from rdkit import Chem
from rdkit.Chem import BRICS
from torch_geometric.data import Data
from tqdm import tqdm
import pandas as pd
from polygraphpy.gnn.pre_processing import PreProcess

class FragmentGA:
    def __init__(self, csv_path, model, preprocess: PreProcess, atom_encoder, bond_encoder, population_size=20):
        self.df = pd.read_csv(csv_path)
        self.model = model.eval()
        self.preprocess = preprocess
        self.atom_encoder = atom_encoder
        self.bond_encoder = bond_encoder
        self.population_size = population_size

        self.fragments = self._extract_fragments()
        self.device = next(model.parameters()).device

    def _extract_fragments(self):
        all_frags = set()
        for smi in self.df['smiles']:
            mol = Chem.MolFromSmiles(smi)
            try:
                frags = BRICS.BRICSDecompose(mol)
                all_frags.update(frags)
            except:
                continue
        return list(all_frags)

    def _build_random_molecule(self):
        """Combine 2–4 random fragments to build a new molecule."""
        for _ in range(100):
            frags = random.sample(self.fragments, k=random.randint(2, 4))
            try:
                new_mol = BRICS.BRICSBuild(frags)
                for mol in new_mol:
                    smi = Chem.MolToSmiles(mol)
                    Chem.SanitizeMol(Chem.MolFromSmiles(smi))
                    return smi
            except:
                continue
        return None

    def _mol_to_data(self, smiles):
        atoms = []
        bonds = []
        
        m1 = Chem.MolFromSmiles(smiles)
        m1 = Chem.AddHs(m1)
        
        atoms = self.preprocess.get_nodes_information(m1, [], chain_size=0)
        df_nodes = pd.DataFrame(atoms)
        nodes_features = pd.DataFrame(self.atom_encoder.transform(df_nodes.drop(['idx'], axis=1)).toarray())
        x = torch.tensor(nodes_features.astype('float32').values)
        
        bonds = self.preprocess.get_bonds_information(m1, [])
        df_bonds = pd.DataFrame(bonds)
        edge_index = torch.tensor([
            df_bonds.begin_idx.to_list() + df_bonds.end_idx.to_list(),
            df_bonds.end_idx.to_list() + df_bonds.begin_idx.to_list()
        ])
        
        edge_attrs = df_bonds[['type', 'is_conjugated', 'is_aromatic']]
        edge_attrs = pd.concat([edge_attrs, edge_attrs.sort_index(ascending=False)])
        edge_attr = torch.tensor(self.bond_encoder.transform(edge_attrs).toarray(), dtype=torch.float32)
        
        edge_weight = torch.tensor([1.0] * edge_index.shape[1], dtype=torch.float32)
        
        mol_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, edge_weight=edge_weight)
        mol_data.validate()
        
        return mol_data
    
    def _evaluate_fitness(self, smiles, target_polarizability):
        try:
            data = self._mol_to_data(smiles)
            if data is None:
                return -1.0
            data = data.to(self.device)
            with torch.no_grad():
                prediction = self.model(data).item()
            # Score is higher if prediction is close to target
            score = -abs(prediction - target_polarizability)
            return score
        except:
            return -1.0

    def run(self, generations=10, target_polarizability=500.0):
        population = [self._build_random_molecule() for _ in range(self.population_size)]
        population = [p for p in population if p is not None]

        for gen in tqdm(range(generations)):
            print(f"Generation {gen + 1}")
            fitness_scores = [
                (smi, self._evaluate_fitness(smi, target_polarizability)) 
                for smi in tqdm(population)
            ]

            # Selection
            fitness_scores.sort(key=lambda x: x[1], reverse=True)
            top_individuals = fitness_scores[:self.population_size // 2]
            if len(top_individuals) == 0:
                print("No valid molecules in this generation. Reinitializing population...")
                population = [self._build_random_molecule() for _ in range(self.population_size)]
                population = [p for p in population if p is not None]
                continue

            # Crossover
            top_frags = set()
            for smi, _ in top_individuals:
                top_frags.update(BRICS.BRICSDecompose(Chem.MolFromSmiles(smi)))
            original_fragments = self.fragments
            self.fragments = list(top_frags)
            new_population = []
            for _ in range(self.population_size):
                new_smi = self._build_random_molecule()
                if new_smi:
                    new_population.append(new_smi)
            self.fragments = original_fragments
            population = new_population

        return fitness_scores

In [ ]:
# Setup
preprocess = PreProcess(
    input_csv='polarizability_data_monomer.csv',
    train_input_data_path='../polygraphpy/data/training_input_data/',
    polymer_type='monomer',
    target='static_polarizability',
    gnn_output_path='./'
)

df = preprocess.run()
atoms_list, bonds_list = preprocess.extract_atoms_and_bonds_features_from_monomer_smiles()
atom_encoder = preprocess.make_encoder(pd.DataFrame(atoms_list).drop_duplicates().reset_index(drop=True))
bond_encoder = preprocess.make_encoder(pd.DataFrame(bonds_list).drop_duplicates().reset_index(drop=True))

model = torch.load('../polygraphpy/data/gnn_output/model_gcn.pt', weights_only=False)
print(model)

Reading GNN input file.
Removing outliers...
Making data standardization...
Extracting unique features from atoms and bonds.


100%|██████████| 9003/9003 [00:03<00:00, 2714.62it/s]


Making feature encoder.
Making feature encoder.
Training data preparation starting. 9003 to go.


9003it [00:40, 221.48it/s]


Training data preparation finished.
Extracting unique features from atoms and bonds.


100%|██████████| 9003/9003 [00:03<00:00, 2843.09it/s]


Making feature encoder.
Making feature encoder.
GCN(
  (conv1): GCNConv(75, 225)
  (conv2): GCNConv(225, 225)
  (conv3): GCNConv(225, 225)
  (lin1): Linear(in_features=225, out_features=225, bias=True)
  (lin2): Linear(in_features=225, out_features=225, bias=True)
  (lin3): Linear(in_features=225, out_features=225, bias=True)
  (output): Linear(in_features=225, out_features=1, bias=True)
)


In [ ]:
ga = FragmentGA(csv_path='polarizability_data_monomer.csv',
                model=model,
                preprocess=preprocess,
                atom_encoder=atom_encoder,
                bond_encoder=bond_encoder,
                population_size=500)

In [ ]:
target_value = 37.2823007951
top_molecules = ga.run(generations=10, target_polarizability=target_value)

# Top 5 candidates
for smi, score in sorted(top_molecules, key=lambda x: x[1], reverse=True)[:5]:
    print(f"SMILES: {smi} | Fitness: {score:.4f}")

  0%|          | 0/10 [00:00<?, ?it/s]

Generation 1


0it [00:00, ?it/s]


No valid molecules in this generation. Reinitializing population...
Generation 2


0it [00:00, ?it/s]


...

0it [00:00, ?it/s]


No valid molecules in this generation. Reinitializing population...
Generation 10


0it [00:00, ?it/s]


No valid molecules in this generation. Reinitializing population...


100%|██████████| 10/10 [00:00<00:00, 31.49it/s]
